# LLM SecureGate Evaluation Notebook

This notebook evaluates the MVP on prompt-injection detection, sensitive entity recall, false positive rate, latency, and readability.

In [ ]:
from pathlib import Path
import re
import time

import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

from securegate.prompt_defense import PromptDefenseEngine
from securegate.response_sanitizer import ResponseSanitizationEngine

In [ ]:
root = Path.cwd().parent
prompt_df = pd.read_csv(root / 'data' / 'prompt_injection_samples.csv')
leak_df = pd.read_csv(root / 'data' / 'sensitive_leak_samples.csv')

prompt_engine = PromptDefenseEngine()
response_engine = ResponseSanitizationEngine()

In [ ]:
prompt_preds = []
prompt_latencies = []

for text in prompt_df['text']:
    started = time.perf_counter()
    result = prompt_engine.inspect(text)
    prompt_latencies.append((time.perf_counter() - started) * 1000)
    prompt_preds.append(1 if result.blocked or result.risk_score >= 0.5 else 0)

prompt_accuracy = accuracy_score(prompt_df['label'], prompt_preds)
prompt_recall = recall_score(prompt_df['label'], prompt_preds)
tn, fp, fn, tp = confusion_matrix(prompt_df['label'], prompt_preds).ravel()
prompt_fpr = fp / (fp + tn) if (fp + tn) else 0.0
prompt_avg_latency_ms = sum(prompt_latencies) / len(prompt_latencies)

print('Prompt Injection Accuracy:', round(prompt_accuracy, 3))
print('Prompt Injection Recall:', round(prompt_recall, 3))
print('Prompt False Positive Rate:', round(prompt_fpr, 3))
print('Prompt Avg Latency (ms):', round(prompt_avg_latency_ms, 3))

In [ ]:
leak_preds = []
leak_latencies = []

for text in leak_df['text']:
    started = time.perf_counter()
    result = response_engine.sanitize(text)
    leak_latencies.append((time.perf_counter() - started) * 1000)
    leak_preds.append(1 if len(result.redactions) > 0 else 0)

sensitive_recall = recall_score(leak_df['contains_sensitive'], leak_preds)
tn, fp, fn, tp = confusion_matrix(leak_df['contains_sensitive'], leak_preds).ravel()
sensitive_fpr = fp / (fp + tn) if (fp + tn) else 0.0
sensitive_avg_latency_ms = sum(leak_latencies) / len(leak_latencies)

print('Sensitive Detection Recall:', round(sensitive_recall, 3))
print('Sensitive Detection False Positive Rate:', round(sensitive_fpr, 3))
print('Sensitive Scan Avg Latency (ms):', round(sensitive_avg_latency_ms, 3))

In [ ]:
def flesch_reading_ease(text: str) -> float:
    sentences = max(1, len(re.findall(r'[.!?]+', text)))
    words = re.findall(r'\w+', text)
    word_count = max(1, len(words))
    syllables = sum(max(1, len(re.findall(r'[aeiouyAEIOUY]+', w))) for w in words)
    return 206.835 - 1.015 * (word_count / sentences) - 84.6 * (syllables / word_count)

example_outputs = [
    'Accessing or revealing API keys violates policy. Use a secrets manager and rotate credentials regularly.',
    'This response may contain sensitive information, so parts were redacted for safety.'
]
scores = [flesch_reading_ease(t) for t in example_outputs]
print('Average Flesch Score:', round(sum(scores) / len(scores), 2))